# Aarya Run — Quantitative Metrics Extraction (All Fault Bucket Directories)

Scans all fault bucket directories under a base output path and extracts
quantitative metrics for every fault in every directory.

**Pipeline:**
- Discovers all `fault_buckets/` subdirectories (or flat directories) containing `raw_trace_*_bucket_*.json`
- Runs `TraceMetricsExtractor` per bucket file
- Summarises results across all runs and faults

## 1. Imports & Path Setup

In [6]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path
from pprint import pprint

# Resolve certifier root
CERTIFIER_ROOT = Path.cwd()
while CERTIFIER_ROOT.name != "certifier" and CERTIFIER_ROOT != CERTIFIER_ROOT.parent:
    CERTIFIER_ROOT = CERTIFIER_ROOT.parent
if str(CERTIFIER_ROOT) not in sys.path:
    sys.path.insert(0, str(CERTIFIER_ROOT))

print(f"Certifier root: {CERTIFIER_ROOT}")

from metrics_extractor.scripts.metrics_extractor_from_trace import TraceMetricsExtractor
from metrics_extractor.schema.metrics_model import LLMQuantitativeExtraction

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)
print("Imports OK.")

Certifier root: c:\Users\meemankgupta\Music\Project\infosys\certifier
Imports OK.


## 2. Fault Buckets Directory & Files

In [7]:
BASE_OUTPUT_DIR = Path(
    r"C:\Users\meemankgupta\Music\Project\infosys\certifier"
    r"\data\output\08-05-26-aarya\1960bc89"
    r"\56fe3250-parallel-defaults"
    r"\56fe3250-4025-495a-ada1-e619d976969b\fault_buckets"
)

# Discover every bucket file in this directory
bucket_dirs = [BASE_OUTPUT_DIR]

print(f"Fault buckets directory: {BASE_OUTPUT_DIR}")
files = sorted(f.name for f in BASE_OUTPUT_DIR.glob("raw_trace_*bucket_*.json") if "manifest" not in f.name)
print(f"Found {len(files)} bucket file(s):")
for name in files:
    print(f"  {name}")

Fault buckets directory: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\08-05-26-aarya\1960bc89\56fe3250-parallel-defaults\56fe3250-4025-495a-ada1-e619d976969b\fault_buckets
Found 3 bucket file(s):
  raw_trace_parallel_bucket_pod-cpu-hog.json
  raw_trace_parallel_bucket_pod-memory-hog.json
  raw_trace_parallel_bucket_pod-network-loss.json


## 3. Extract Quantitative Metrics — All Directories × All Faults

Results are stored as `results[(run_label, fault_name)] = LLMQuantitativeExtraction`.

In [8]:
results = {}  # fault_name -> LLMQuantitativeExtraction
errors = {}   # fault_name -> error message

bucket_files = sorted(
    f for f in BASE_OUTPUT_DIR.glob("raw_trace_*bucket_*.json")
    if "manifest" not in f.name
)

print(f"Processing {len(bucket_files)} fault bucket(s) from:\n  {BASE_OUTPUT_DIR}\n")

for bucket_file in bucket_files:
    fault_name = bucket_file.stem.split("bucket_", 1)[-1]  # pod-cpu-hog etc.

    print(f"\n{'='*60}")
    print(f"Fault: {fault_name}")
    print(f"{'='*60}")

    try:
        extractor = TraceMetricsExtractor()
        spans = extractor.load_trace_file(str(bucket_file))
        print(f"  spans={len(spans)}  batches={len(extractor._create_batches(spans))}")
        print(f"  injection_timestamp: {extractor.bucket_metadata.get('injection_timestamp')}")
        print(f"  fault_name (ground truth): {extractor.bucket_metadata.get('fault_name')}")

        quant = await extractor.extract_quantitative_metrics(spans)
        results[fault_name] = quant

        print(f"\n  --- Extracted Metrics ---")
        print(f"  detection_success:          {quant.detection_success}")
        print(f"  fault_detected:             {quant.fault_detected}")
        print(f"  detected_fault_type:        {quant.detected_fault_type}")
        print(f"  fault_target_service:       {quant.fault_target_service}")
        print(f"  fault_namespace:            {quant.fault_namespace}")
        print(f"  fault_injection_time:       {quant.fault_injection_time}")
        print(f"  agent_fault_detection_time: {quant.agent_fault_detection_time}")
        print(f"  agent_fault_mitigation_time:{quant.agent_fault_mitigation_time}")
        print(f"  time_to_detect (s):         {quant.time_to_detect}")
        print(f"  time_to_mitigate (s):       {quant.time_to_mitigate}")
        print(f"  input_tokens:               {quant.input_tokens}")
        print(f"  output_tokens:              {quant.output_tokens}")
        print(f"  trajectory_steps:           {quant.trajectory_steps}")
        print(f"  tool_calls:                 {len(quant.tool_calls)}")
        print(f"  tool_selection_accuracy:    {quant.tool_selection_accuracy}")
        print(f"  pii_detection:              {quant.pii_detection}")
    except Exception as e:
        errors[fault_name] = str(e)
        print(f"  ERROR: {e}")

print(f"\nDone. {len(results)} succeeded, {len(errors)} failed.")
if errors:
    print("Failures:")
    for k, msg in errors.items():
        print(f"  {k}: {msg}")

2026-05-11 21:09:17,804 - [metrics_extractor_from_trace.py : load_trace_file : 286] - INFO - Loaded bucket metadata from trace file: fault_id=pod-cpu-hog, fault_name=pod-cpu-hog


Processing 3 fault bucket(s) from:
  C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\08-05-26-aarya\1960bc89\56fe3250-parallel-defaults\56fe3250-4025-495a-ada1-e619d976969b\fault_buckets


Fault: pod-cpu-hog
  spans=12  batches=2
  injection_timestamp: 2026-05-08T07:57:53.000Z
  fault_name (ground truth): pod-cpu-hog


2026-05-11 21:09:18,732 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: embedding_model
2026-05-11 21:09:19,099 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: gpt-4o
2026-05-11 21:09:19,422 - [azure_openai_util.py : get_clients : 101] - INFO - Created Azure OpenAI client for model: gpt-5.2
2026-05-11 21:09:19,423 - [azure_openai_util.py : __init__ : 44] - INFO - AzureLLMClient initialized successfully
2026-05-11 21:09:19,424 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 705] - INFO - Processing 12 spans in 2 batches
2026-05-11 21:09:19,425 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 1/2
2026-05-11 21:09:19,431 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:09:51,095 - [metrics_extractor_from_trace.p


  --- Extracted Metrics ---
  detection_success:          1
  fault_detected:             User service pod user-5bc44bf4bf-2b47s in namespace sock-shop shows repeated readiness and liveness probe failures (HTTP /health timeouts and connection reset), resulting in container restarts and degraded health for the user component.
  detected_fault_type:        None
  fault_target_service:       name=carts
  fault_namespace:            sock-shop
  fault_injection_time:       2026-05-08T07:57:53.000Z
  agent_fault_detection_time: 2026-05-08T08:00:42.270Z
  agent_fault_mitigation_time:None
  time_to_detect (s):         169.27
  time_to_mitigate (s):       None
  input_tokens:               5445
  output_tokens:              2457
  trajectory_steps:           12
  tool_calls:                 83
  tool_selection_accuracy:    0.7742
  pii_detection:              True

Fault: pod-memory-hog
  spans=12  batches=2
  injection_timestamp: 2026-05-08T07:57:54.000Z
  fault_name (ground truth): pod-memor

2026-05-11 21:10:36,586 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 2/2
2026-05-11 21:10:36,591 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:11:04,439 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 715] - INFO - Aggregating quantitative metrics from all batches
2026-05-11 21:11:04,441 - [metrics_extractor_from_trace.py : _aggregate_quantitative_metrics : 622] - INFO - Identifying detection and mitigation spans using LLM...
2026-05-11 21:11:12,740 - [metrics_extractor_from_trace.py : _identify_detection_mitigation_spans : 386] - INFO - LLM identified detection span: time-08-00-42-270713_chatcmpl-DdAQUU7kB58Hn1DjRIu9Osaue1upN at 2026-05-08T08:00:42.270Z
2026-05-11 21:11:12,914 - [metrics_extractor_from_trace.py : _aggregate_quantitative_metrics : 627] - INFO - PII pre-scan: detected=False,


  --- Extracted Metrics ---
  detection_success:          1
  fault_detected:             User service pod user-5bc44bf4bf-2b47s in namespace sock-shop is experiencing repeated readiness and liveness probe failures on /health with timeouts, EOF, and connection reset by peer, causing restarts. All other pods including orders in sock-shop are Running with 1/1 or 2/2 containers ready.
  detected_fault_type:        None
  fault_target_service:       name=orders
  fault_namespace:            sock-shop
  fault_injection_time:       2026-05-08T07:57:54.000Z
  agent_fault_detection_time: 2026-05-08T08:00:42.270Z
  agent_fault_mitigation_time:None
  time_to_detect (s):         168.27
  time_to_mitigate (s):       None
  input_tokens:               5445
  output_tokens:              2457
  trajectory_steps:           12
  tool_calls:                 83
  tool_selection_accuracy:    0.9565
  pii_detection:              True

Fault: pod-network-loss
  spans=26  batches=5
  injection_timestamp: 20

2026-05-11 21:12:02,562 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 2/5
2026-05-11 21:12:02,572 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:13:20,615 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 3/5
2026-05-11 21:13:20,622 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:14:18,782 - [metrics_extractor_from_trace.py : extract_quantitative_metrics : 709] - INFO - Processing quantitative batch 4/5
2026-05-11 21:14:18,791 - [azure_openai_util.py : _get_or_create_agent : 135] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-11 21:15:34,949 - [metrics_extractor_from_trace.py : extract_quantit


  --- Extracted Metrics ---
  detection_success:          0
  fault_detected:             User service degradation in namespace sock-shop: user deployment has 0 available replicas, user pod user-5bc44bf4bf-2b47s is not Ready, repeatedly failing readiness and liveness probes to /health with timeouts, EOF, connection reset by peer, and connection refused, and logs show persistent 'no reachable servers' errors when connecting to its backend (configured via MONGO_HOST=user-db:27017). All other sock-shop pods and deployments, including user-db and other microservices, are Running and Ready. No events mention chaos, inject, fault, or disrupt; flash-agent deployment appears healthy and only shows normal lifecycle events.
  detected_fault_type:        pod-network-loss
  fault_target_service:       name=user-db
  fault_namespace:            sock-shop
  fault_injection_time:       2026-05-08T07:57:54.000Z
  agent_fault_detection_time: 2026-05-08T07:59:13Z
  agent_fault_mitigation_time:None
  ti

## 4. Summary Table — All 3 Faults

In [9]:
print(f"{'Fault':<22} {'Det':>5} {'TTD(s)':>9} {'TTM(s)':>9} {'In-Tok':>9} {'Out-Tok':>9} {'Tools':>6} {'ToolAcc':>9}")
print("-" * 82)
for fault_name, q in sorted(results.items()):
    print(
        f"{fault_name:<22} "
        f"{str(q.detection_success):>5} "
        f"{str(round(q.time_to_detect, 1)) if q.time_to_detect else 'N/A':>9} "
        f"{str(round(q.time_to_mitigate, 1)) if q.time_to_mitigate else 'N/A':>9} "
        f"{q.input_tokens or 0:>9} "
        f"{q.output_tokens or 0:>9} "
        f"{len(q.tool_calls):>6} "
        f"{str(round(q.tool_selection_accuracy, 3)) if q.tool_selection_accuracy else 'N/A':>9}"
    )

Fault                    Det    TTD(s)    TTM(s)    In-Tok   Out-Tok  Tools   ToolAcc
----------------------------------------------------------------------------------
pod-cpu-hog                1     169.3       N/A      5445      2457     83     0.774
pod-memory-hog             1     168.3       N/A      5445      2457     83     0.957
pod-network-loss           0      79.0       N/A     56134     19356    105     0.632


## 5. Full JSON Output Per Fault

In [10]:
for fault_name, q in results.items():
    print(f"\n{'='*60}")
    print(f"FULL JSON — {fault_name}")
    print(f"{'='*60}")
    print(q.model_dump_json(indent=2))


FULL JSON — pod-cpu-hog
{
  "agent_name": "demoinfra2",
  "agent_id": "1960bc89-361a-4fcb-af86-699113f09ec9",
  "agent_version": "3.0.0",
  "experiment_id": "56fe3250-4025-495a-ada1-e619d976969b",
  "run_id": "859897c9-82dd-4a35-bfb4-33c93f235ac2",
  "fault_injection_time": "2026-05-08T07:57:53.000Z",
  "agent_fault_detection_time": "2026-05-08T08:00:42.270Z",
  "agent_fault_mitigation_time": null,
  "time_to_detect": 169.27,
  "time_to_mitigate": null,
  "fault_detected": "User service pod user-5bc44bf4bf-2b47s in namespace sock-shop shows repeated readiness and liveness probe failures (HTTP /health timeouts and connection reset), resulting in container restarts and degraded health for the user component.",
  "detection_success": 1,
  "trajectory_steps": 12,
  "input_tokens": 5445,
  "output_tokens": 2457,
  "injected_fault_name": "pod-cpu-hog",
  "injected_fault_category": "resource",
  "detected_fault_type": null,
  "fault_target_service": "name=carts",
  "fault_namespace": "sock-s